# Chat with PDF - Notebook Ingestion and Query Pipeline

This notebook demonstrates a complete RAG ingestion and querying pipeline with the following upgrades:
1. **Portability**: Uses sys.argv or env variables to specify the PDF path (no hardcoded path).
2. **Reusability**: Checks if the document source metadata is already in Pinecone before embedding/indexing.
3. **Source Citations**: RAG responses print page numbers and source filenames for each chunk.
4. **Multi-Format Loader**: Supports PDF, DOCX, and TXT files.
5. **Robust API Calls**: Added retries and timeouts to the LLM configurations.
6. **Conversational Chat**: Maintained stateful chat memory to ask follow-up questions.

In [ ]:
# Install dependencies
%pip install langchain-pinecone pinecone-client sentence-transformers docx2txt pypdf python-dotenv langchain-openai langchain-community langchain-core

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

## 1. Load Document (Multi-Format Support)

In [ ]:
import os
import sys
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def load_any(file_path):
    ext = file_path.lower().split('.')[-1]
    loaders = {
        'pdf': PyPDFLoader,
        'docx': Docx2txtLoader,
        'txt': TextLoader,
    }
    if ext not in loaders:
        raise ValueError(f"Unsupported file type: {ext}")
    return loaders[ext](file_path).load()

# Command-line argument, env var, or default path
PDF_PATH = sys.argv[1] if len(sys.argv) > 1 and not sys.argv[0].endswith('ipykernel_launcher.py') else "O2h_resume.pdf"

# Fallback default path if the file is not found locally
if not os.path.exists(PDF_PATH) and PDF_PATH == "O2h_resume.pdf":
    original_path = r"D:\obssessed\PDFS\Gen-Ai\7_document_Loader\O2h_resume.pdf"
    if os.path.exists(original_path):
        PDF_PATH = original_path

print(f"Loading document from: {PDF_PATH}")
docs = load_any(PDF_PATH)
print(f"Loaded {len(docs)} pages/sections.")

## 2. Split Document into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

## 3. Generate Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings model loaded.")

## 4. Pinecone Vector Store (With Reusability Check)

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = 'chatpdf'

# Create index only if it doesn't exist
if index_name not in [i.name for i in pc.list_indexes()]:
    print(f"Creating index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=384,  # all-MiniLM-L6-v2 output dim
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
else:
    print(f"Index '{index_name}' already exists.")

vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# Normalize the file path for metadata checking
norm_path = os.path.abspath(PDF_PATH)

# Only embed if this document hasn't been ingested yet (check by source metadata)
def is_already_indexed(source_path):
    try:
        index = pc.Index(index_name)
        # Query with metadata filter to check if this document's chunks exist
        results = index.query(
            vector=[0.0]*384,
            top_k=1,
            filter={"source": {"$eq": source_path}},
            include_metadata=True
        )
        return len(results.get('matches', [])) > 0
    except Exception as e:
        print(f"Metadata check failed (index may be empty): {e}")
        return False

if not is_already_indexed(norm_path):
    # Update chunk metadata to store the normalized absolute path
    for chunk in chunks:
        chunk.metadata["source"] = norm_path
    vectorstore.add_documents(chunks)
    print("Indexed new document.")
else:
    print("Document already indexed, skipping embedding and upsert.")

## 5. Setup Retriever & LLM (With Retry & Error Handling)

In [ ]:
import time
from langchain_openai import ChatOpenAI

# Retrieve only chunks belonging to this document
retriever = vectorstore.as_retriever(
    search_kwargs={
        'k': 4,
        'filter': {"source": norm_path}
    }
)

def get_llm():
    return ChatOpenAI(
        model="openai/gpt-oss-120b:free",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1",
        max_retries=3,
        timeout=30,
    )

llm = get_llm()
print("Retriever and LLM initialized.")

## 6. Prompt, Output Parser, and Chain (With History & Citations)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

def format_docs_with_sources(docs):
    formatted = []
    for i, doc in enumerate(docs):
        # Parse page number (PyPDFLoader uses page starting from 0)
        page = doc.metadata.get('page_label')
        if page is None and 'page' in doc.metadata:
            page = int(doc.metadata['page']) + 1
        page_str = f"Page {page}" if page is not None else "Unknown Page"
        
        # Format filename
        filename = os.path.basename(doc.metadata.get('source', 'Document'))
        formatted.append(f"[Source {i+1}: {filename} - {page_str}]\n{doc.page_content}")
    return "\n\n".join(formatted)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context. Cite the source number(s) you used, like [Source 1], [Source 2], etc. If the answer is not in the context, say 'I don't know.'\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

parser = StrOutputParser()

chain = (
    {
        "context": lambda x: format_docs_with_sources(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm
    | parser
)

chat_history = []

def safe_invoke(chain, inputs, retries=2):
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            if attempt == retries - 1:
                return f"Error: Could not get a response ({e})"
            time.sleep(2 ** attempt)

def ask(question):
    response = safe_invoke(chain, {"question": question, "chat_history": chat_history})
    if not response.startswith("Error:"):
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=response))
    return response

In [ ]:
# Test query
response = ask("What is technical skill list out all of them?")
print(response)

In [ ]:
# Follow up query to test chat history
response = ask("What other projects are listed there?")
print(response)

In [ ]:
try:
    chain.get_graph().print_ascii()
except Exception as e:
    print(f"Could not print ascii graph: {e}")